# American Express default prediction workflow

Follow the customer-level pipeline from statement histories to validation and submission. The markdown explains the flow; the code cells run the corresponding stages when the local files are ready.

## 1 Load the configuration

Install the project with `python -m pip install -e ".[dev,notebook]"` from the project root. Use that environment as the notebook kernel. Paths are resolved relative to the project root.

In [ ]:
from pathlib import Path
import json
from amex_default.config import load_config, plan

ROOT = Path.cwd()
if not (ROOT / "configs").is_dir():
    ROOT = ROOT.parent
cfg = load_config(ROOT / "configs" / "baseline.json")
print(json.dumps(plan(cfg), indent=2))

## 2 Turn statements into customer features

Each customer has up to 13 statements. For a numerical series such as 0.2, 0.4, 0.8, the mean is about 0.467, the minimum is 0.2, the maximum is 0.8, and the last value is 0.8. These summaries capture both the history and the latest state.

The baseline generates mean, min, max, and chronological last for numerical columns. It also generates last and unique count for categorical columns, plus statement count and history span. DuckDB partitions statements by customer before aggregation.

In [ ]:
from amex_default.features import prepare

train_metadata = prepare(cfg, "train")
train_metadata["statistics"]

## 3 Train and validate by customer

Five stratified folds split customer rows. Each customer receives a prediction from a model trained on other customers. Labels are aligned by ID, and each fold learns its category vocabulary from its own training rows. LightGBM uses AMEX for early stopping.

Training saves pooled OOF metrics and the mean of fold AMEX scores separately. The supplied reference Quickstart reports a mean fold AMEX of 0.79383 using 469 features on denoised Parquet data.

In [ ]:
from amex_default.training import train

run_metrics = train(cfg)
run_metrics

## 4 Read ranking and probability diagnostics

AMEX combines weighted Gini with default capture at a 4% cumulative-weight cutoff. ROC-AUC and average precision summarize ranking. Calibration, log loss, and Brier score assess predicted probabilities in the sampled competition population.

In [ ]:
from amex_default.reporting import report

report(cfg)

## 5 Prepare test features and predict

The same feature recipe is applied to test statements. Each fold reuses its saved category vocabulary. The ensemble averages raw margins and applies sigmoid, then joins predictions to the sample submission by customer ID.

In [ ]:
from amex_default.prediction import predict

prepare(cfg, "test")
predict(cfg)

## 6 Inspect history coverage and missingness

The data profile summarizes statement counts and customer feature missingness. Compare the train and test columns in `feature_missingness.csv` to find changes in information availability. The supplied EDA highlights B_29 as an example.

In [ ]:
from amex_default.reporting import profile

profile(cfg)

## 7 Extend the experiment

Use `configs/extended.json` for standard deviations, first values, first-to-last changes, last-minus-mean differences, and missing fractions. It stores its features and outputs in separate experiment folders. Compare the two runs using the same five customer folds and seed.

Read `docs/methodology.md` for precise definitions and `02_results.ipynb` for output inspection.